# mt_retrieval — Semantic Retrieval Module

In-memory vector store over the semantic knowledge layer. Loaded once by `00_main`.

| Parameter | Value |
|---|---|
| Embedding model | `databricks-gte-large-en` (1024-dim) |
| Chunk strategy | Object-level (one chunk per table/metric/rule/join) |
| Chunks | 53 total (10 table, 8 column, 11 metric, 10 join, 14 business rule) |
| Top-k | 5 |
| Similarity threshold | 0.3 |
| Vector store | In-memory numpy (cosine similarity) |

Provides `retrieve_semantic_context(question)` used by SAS_RAG, MAS_RAG, and DYNAMIC_FILTERED_MAS_RAG.

In [0]:
import numpy as np
import time
import hashlib
from typing import List, Dict, Optional

# Uses RETRIEVAL_PARAMS from mt_config (loaded before this notebook)
_EMBEDDING_MODEL = RETRIEVAL_PARAMS["embedding_model"]
_DEFAULT_TOP_K = RETRIEVAL_PARAMS["top_k"]
_DEFAULT_THRESHOLD = RETRIEVAL_PARAMS["similarity_threshold"]

print(f"mt_retrieval: embedding_model={_EMBEDDING_MODEL}, top_k={_DEFAULT_TOP_K}, threshold={_DEFAULT_THRESHOLD}")

In [0]:
# ============================================================
# EMBEDDING CLIENT — uses Databricks Foundation Model API
# ============================================================

def _get_embeddings(texts: List[str]) -> np.ndarray:
    """
    Embed a list of texts using the Databricks embedding endpoint.
    Returns numpy array of shape (len(texts), embedding_dim).
    Batches requests in groups of 16 to avoid payload limits.
    """
    from openai import OpenAI

    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    client = OpenAI(api_key=token, base_url=f"https://{workspace_url}/serving-endpoints")

    all_embeddings = []
    batch_size = 16

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=_EMBEDDING_MODEL, input=batch)
        batch_embs = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embs)

    return np.array(all_embeddings, dtype=np.float32)


def _cosine_similarity(query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    """Cosine similarity between one query vector and many document vectors."""
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-10)
    doc_norms = doc_vecs / (np.linalg.norm(doc_vecs, axis=1, keepdims=True) + 1e-10)
    return doc_norms @ query_norm


print("✓ Embedding client ready")

In [0]:
# ============================================================
# CHUNK BUILDER — Creates semantic knowledge documents for embedding
# ============================================================
# Each chunk represents one semantic object (table, metric, rule, relationship).
# Chunk text is optimized for retrieval relevance.
# ============================================================

def _build_semantic_chunks() -> List[Dict]:
    """
    Load semantic layer tables and create one chunk per semantic object.
    Returns list of dicts: {chunk_id, chunk_type, metadata, text}
    """
    # Ensure Spark session is accessible (needed for .ipynb on serverless)
    from pyspark.sql import SparkSession
    _spark = SparkSession.getActiveSession()
    if _spark is None:
        try:
            _spark = SparkSession.builder.getOrCreate()
        except Exception:
            pass
    if _spark is None:
        # Last resort: use globals
        _spark = globals().get('spark')
    if _spark is None:
        print("  \u26a0 No Spark session available. Cannot build semantic chunks.")
        return []

    chunks = []
    schema = f"{MT_CATALOG}.{MT_SCHEMA}"

    # --- 1. Table metadata chunks ---
    try:
        rows = _spark.sql(f"SELECT * FROM {schema}.mt_semantic_table_metadata").collect()
        for r in rows:
            chunk_id = f"table_{r.table_name}"
            text = (
                f"TABLE: {schema}.{r.table_name}\n"
                f"Business name: {r.business_name or 'N/A'}\n"
                f"Description: {r.business_description or 'N/A'}\n"
                f"Grain: {r.grain or 'N/A'}\n"
                f"Business key: {r.business_key or 'N/A'}\n"
                f"Domain: {r.business_domain or 'N/A'}\n"
                f"Primary use cases: {r.primary_use_cases or 'general'}"
            )
            chunks.append({
                "chunk_id": chunk_id,
                "chunk_type": "table_metadata",
                "metadata": {"table_name": r.table_name},
                "text": text
            })
    except Exception as e:
        print(f"  ⚠ Could not load table metadata: {e}")

    # --- 2. Metric definition chunks ---
    try:
        rows = _spark.sql(f"SELECT * FROM {schema}.mt_semantic_metric_definitions").collect()
        for r in rows:
            chunk_id = f"metric_{r.metric_name}" if hasattr(r, 'metric_name') else f"metric_{hashlib.md5(str(r).encode()).hexdigest()[:8]}"
            name = r.metric_name if hasattr(r, 'metric_name') else r.name if hasattr(r, 'name') else 'unknown'
            text = (
                f"METRIC: {name}\n"
                f"Definition: {r.definition if hasattr(r, 'definition') else r.description if hasattr(r, 'description') else 'N/A'}\n"
                f"SQL formula: {r.sql_formula if hasattr(r, 'sql_formula') else r.formula if hasattr(r, 'formula') else 'N/A'}\n"
                f"Tables used: {r.tables_used if hasattr(r, 'tables_used') else 'N/A'}"
            )
            chunks.append({
                "chunk_id": chunk_id,
                "chunk_type": "metric_definition",
                "metadata": {"metric_name": name},
                "text": text
            })
    except Exception as e:
        print(f"  ⚠ Could not load metric definitions: {e}")

    # --- 3. Business rule chunks ---
    try:
        rows = _spark.sql(f"SELECT * FROM {schema}.mt_semantic_business_rules").collect()
        for r in rows:
            rule_name = r.rule_name if hasattr(r, 'rule_name') else r.name if hasattr(r, 'name') else 'unknown'
            chunk_id = f"rule_{rule_name}"
            text = (
                f"BUSINESS RULE: {rule_name}\n"
                f"Description: {r.description if hasattr(r, 'description') else r.rule_text if hasattr(r, 'rule_text') else 'N/A'}\n"
                f"Application: {r.application if hasattr(r, 'application') else r.context if hasattr(r, 'context') else 'N/A'}"
            )
            chunks.append({
                "chunk_id": chunk_id,
                "chunk_type": "business_rule",
                "metadata": {"rule_name": rule_name},
                "text": text
            })
    except Exception as e:
        print(f"  ⚠ Could not load business rules: {e}")

    # --- 4. Join rule / relationship chunks ---
    try:
        rows = _spark.sql(f"SELECT * FROM {schema}.mt_semantic_join_rules").collect()
        for r in rows:
            rel_name = r.relationship_name if hasattr(r, 'relationship_name') else r.join_name if hasattr(r, 'join_name') else 'unknown'
            chunk_id = f"join_{rel_name}"
            text = (
                f"JOIN RELATIONSHIP: {rel_name}\n"
                f"Tables: {r.left_table if hasattr(r, 'left_table') else 'N/A'} ↔ {r.right_table if hasattr(r, 'right_table') else 'N/A'}\n"
                f"Join condition: {r.join_condition if hasattr(r, 'join_condition') else r.condition if hasattr(r, 'condition') else 'N/A'}\n"
                f"Description: {r.description if hasattr(r, 'description') else 'N/A'}"
            )
            chunks.append({
                "chunk_id": chunk_id,
                "chunk_type": "join_rule",
                "metadata": {"relationship": rel_name},
                "text": text
            })
    except Exception as e:
        print(f"  ⚠ Could not load join rules: {e}")

    # --- 5. Column metadata chunks (grouped by table) ---
    try:
        rows = _spark.sql(f"SELECT * FROM {schema}.mt_semantic_column_metadata").collect()
        # Group by table
        table_cols = {}
        for r in rows:
            tbl = r.table_name if hasattr(r, 'table_name') else 'unknown'
            if tbl not in table_cols:
                table_cols[tbl] = []
            col_name = r.column_name if hasattr(r, 'column_name') else 'unknown'
            col_desc = r.description if hasattr(r, 'description') else r.business_description if hasattr(r, 'business_description') else ''
            table_cols[tbl].append(f"  - {col_name}: {col_desc}")

        for tbl, cols in table_cols.items():
            chunk_id = f"columns_{tbl}"
            text = f"COLUMNS FOR TABLE {schema}.{tbl}:\n" + "\n".join(cols[:20])  # cap at 20 columns
            chunks.append({
                "chunk_id": chunk_id,
                "chunk_type": "column_metadata",
                "metadata": {"table_name": tbl},
                "text": text
            })
    except Exception as e:
        print(f"  ⚠ Could not load column metadata: {e}")

    return chunks


print("✓ Chunk builder defined")

In [0]:
# ============================================================
# VECTOR STORE — In-memory numpy-based semantic search
# ============================================================

class SemanticVectorStore:
    """In-memory vector store for semantic knowledge retrieval."""

    def __init__(self):
        self.chunks: List[Dict] = []
        self.embeddings: Optional[np.ndarray] = None
        self._built = False

    def build(self):
        """Build the vector store: load chunks, compute embeddings."""
        print("Building semantic vector store...")
        start = time.time()

        self.chunks = _build_semantic_chunks()
        if not self.chunks:
            print("  ⚠ No semantic chunks found. RAG retrieval will return empty results.")
            self._built = True
            return

        texts = [c["text"] for c in self.chunks]
        self.embeddings = _get_embeddings(texts)
        self._built = True

        elapsed = time.time() - start
        print(f"  ✓ Vector store built: {len(self.chunks)} chunks, "
              f"{self.embeddings.shape[1]}-dim embeddings, {elapsed:.1f}s")
        # Summary by type
        from collections import Counter
        type_counts = Counter(c["chunk_type"] for c in self.chunks)
        for ct, cnt in sorted(type_counts.items()):
            print(f"    {ct}: {cnt}")

    def query(
        self,
        question: str,
        top_k: int = _DEFAULT_TOP_K,
        threshold: float = _DEFAULT_THRESHOLD
    ) -> List[Dict]:
        """
        Retrieve top-k most relevant semantic chunks for a question.

        Returns list of dicts:
          {chunk_id, rank, similarity_score, chunk_type, metadata, text}
        Only chunks above similarity_threshold are returned.
        """
        if not self._built:
            self.build()

        if self.embeddings is None or len(self.chunks) == 0:
            return []

        # Embed the question
        q_emb = _get_embeddings([question])[0]

        # Compute cosine similarities
        scores = _cosine_similarity(q_emb, self.embeddings)

        # Rank and filter
        ranked_indices = np.argsort(scores)[::-1]
        results = []
        for rank, idx in enumerate(ranked_indices[:top_k], start=1):
            sim = float(scores[idx])
            if sim < threshold:
                break
            chunk = self.chunks[idx]
            results.append({
                "chunk_id": chunk["chunk_id"],
                "rank": rank,
                "similarity_score": round(sim, 4),
                "chunk_type": chunk["chunk_type"],
                "metadata": chunk["metadata"],
                "text": chunk["text"],
            })

        return results


# Singleton instance
_vector_store = SemanticVectorStore()

print("✓ SemanticVectorStore class defined")

In [0]:
# ============================================================
# PUBLIC API — Used by SAS_RAG, MAS_RAG, DYNAMIC_FILTERED_MAS_RAG
# ============================================================

def retrieve_semantic_context(
    question: str,
    top_k: int = _DEFAULT_TOP_K,
    threshold: float = _DEFAULT_THRESHOLD
) -> List[Dict]:
    """
    Retrieve the most relevant semantic knowledge chunks for a question.

    Args:
        question: natural language question
        top_k: max chunks to return (default from RETRIEVAL_PARAMS)
        threshold: minimum cosine similarity (default from RETRIEVAL_PARAMS)

    Returns:
        List of dicts with: chunk_id, rank, similarity_score, chunk_type, metadata, text
    """
    return _vector_store.query(question, top_k=top_k, threshold=threshold)


def format_retrieved_context_for_prompt(chunks: List[Dict]) -> str:
    """
    Format retrieved chunks into a prompt-friendly string.
    Used by agent prompts to inject retrieved semantic knowledge.
    """
    if not chunks:
        return "(No relevant semantic knowledge retrieved)"

    lines = ["RETRIEVED SEMANTIC KNOWLEDGE (ranked by relevance):"]
    for c in chunks:
        lines.append(f"\n[{c['rank']}] ({c['chunk_type']}, score={c['similarity_score']:.3f})")
        lines.append(c["text"])
    return "\n".join(lines)


def get_retrieval_log(chunks: List[Dict]) -> List[Dict]:
    """
    Format retrieved chunks for logging (strips full text, keeps metadata).
    """
    return [
        {
            "chunk_id": c["chunk_id"],
            "rank": c["rank"],
            "similarity_score": c["similarity_score"],
            "chunk_type": c["chunk_type"],
            "metadata": c["metadata"],
            "text_preview": c["text"][:150]
        }
        for c in chunks
    ]


print("✓ Public API ready: retrieve_semantic_context(), format_retrieved_context_for_prompt(), get_retrieval_log()")

In [0]:
# Build vector store on %run ./mt_retrieval
_vector_store.build()

print(f"\n✓ mt_retrieval loaded and ready")
print(f"  retrieve_semantic_context(question) → top-{_DEFAULT_TOP_K} chunks (threshold={_DEFAULT_THRESHOLD})")
print(f"  format_retrieved_context_for_prompt(chunks) → prompt string")
print(f"  get_retrieval_log(chunks) → logging-friendly list")